In [0]:
from pyspark.sql import functions as F

# Create Gold schema
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

# Load Silver tables
orders = spark.table("workspace.silver.orders")
order_items = spark.table("workspace.silver.order_items")
customers = spark.table("workspace.silver.customers")
products = spark.table("workspace.silver.products")
sellers = spark.table("workspace.silver.sellers")
payments = spark.table("workspace.silver.payments")
reviews = spark.table("workspace.silver.reviews")
category_translation = spark.table(
    "workspace.silver.category_translation"
)

print("Silver datasets loaded successfully!")

Silver datasets loaded successfully!


In [0]:
# Customer dimension
dim_customers = customers.select(
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
)

# Product dimension
dim_products = (
    products
    .join(
        category_translation,
        products.product_category_name ==
        category_translation.product_category_name,
        "left"
    )
    .select(
        products["product_id"],
        products["product_category_name"],
        category_translation["product_category_name_english"],
        products["product_name_lenght"],
        products["product_description_lenght"],
        products["product_photos_qty"],
        products["product_weight_g"],
        products["product_length_cm"],
        products["product_height_cm"],
        products["product_width_cm"]
    )
)

# Seller dimension
dim_sellers = sellers.select(
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state"
)

# Date dimension based on order purchase dates
dim_date = (
    orders
    .select(
        F.to_date("order_purchase_timestamp").alias("date")
    )
    .filter(F.col("date").isNotNull())
    .distinct()
    .withColumn("year", F.year("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("day_of_week", F.dayofweek("date"))
)

# Save dimensions
dimensions = {
    "dim_customers": dim_customers,
    "dim_products": dim_products,
    "dim_sellers": dim_sellers,
    "dim_date": dim_date
}

for name, df in dimensions.items():
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.gold.{name}")
    )

print("All dimension tables created!")

All dimension tables created!


In [0]:
# Aggregate order items separately to prevent join fanout
item_aggregates = (
    order_items
    .groupBy("order_id")
    .agg(
        F.count("*").alias("total_items"),
        F.sum("price").alias("total_item_value"),
        F.sum("freight_value").alias("total_freight")
    )
)

# Aggregate payments separately
payment_aggregates = (
    payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("total_payment"),
        F.count("*").alias("payment_count"),
        F.concat_ws(
            ", ",
            F.sort_array(F.collect_set("payment_type"))
        ).alias("payment_types")
    )
)

# Aggregate reviews separately
review_aggregates = (
    reviews
    .groupBy("order_id")
    .agg(
        F.avg("review_score").alias("average_review_score"),
        F.count("*").alias("review_count")
    )
)

# Fact orders: one row per order
fact_orders = (
    orders
    .join(item_aggregates, "order_id", "left")
    .join(payment_aggregates, "order_id", "left")
    .join(review_aggregates, "order_id", "left")
    .withColumn(
        "delivery_days",
        F.datediff(
            F.to_date("order_delivered_customer_date"),
            F.to_date("order_purchase_timestamp")
        )
    )
    .withColumn(
        "is_late_delivery",
        F.when(
            F.col("order_delivered_customer_date").isNotNull() &
            F.col("order_estimated_delivery_date").isNotNull(),
            F.to_date("order_delivered_customer_date") >
            F.to_date("order_estimated_delivery_date")
        ).otherwise(None)
    )
    .withColumn(
        "purchase_date",
        F.to_date("order_purchase_timestamp")
    )
)

# Fact order items: one row per order item
fact_order_items = (
    order_items
    .join(
        orders.select(
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ),
        "order_id",
        "left"
    )
    .join(
        products.select(
            "product_id",
            "product_category_name"
        ),
        "product_id",
        "left"
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            F.to_date("order_delivered_customer_date"),
            F.to_date("order_purchase_timestamp")
        )
    )
    .withColumn(
        "is_late_delivery",
        F.when(
            F.col("order_delivered_customer_date").isNotNull() &
            F.col("order_estimated_delivery_date").isNotNull(),
            F.to_date("order_delivered_customer_date") >
            F.to_date("order_estimated_delivery_date")
        ).otherwise(None)
    )
)

# Save fact tables
(
    fact_orders.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.fact_orders")
)

(
    fact_order_items.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.fact_order_items")
)

print("Fact tables created successfully!")

Fact tables created successfully!


In [0]:
# Daily sales analytics
daily_sales = (
    fact_orders
    .groupBy("purchase_date")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("total_item_value").alias("total_revenue"),
        F.sum("total_freight").alias("total_freight"),
        F.avg("total_item_value").alias("average_order_value"),
        F.sum(
            F.when(F.col("order_status") == "delivered", 1)
            .otherwise(0)
        ).alias("delivered_orders"),
        F.sum(
            F.when(F.col("is_late_delivery") == True, 1)
            .otherwise(0)
        ).alias("late_orders")
    )
)

# Seller performance analytics
seller_performance = (
    fact_order_items
    .groupBy("seller_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("price").alias("total_revenue"),
        F.avg("price").alias("average_item_price"),
        F.countDistinct(
            F.when(
                F.col("is_late_delivery") == True,
                F.col("order_id")
            )
        ).alias("late_orders")
    )
)

# Delivery performance analytics
delivery_performance = (
    fact_orders
    .groupBy("order_status")
    .agg(
        F.count("*").alias("total_orders"),

        F.sum(
            F.when(
                F.col("delivery_days").isNotNull(), 1
            ).otherwise(0)
        ).alias("orders_with_delivery_dates"),

        F.sum(
            F.when(
                F.col("delivery_days").isNull(), 1
            ).otherwise(0)
        ).alias("orders_without_delivery_dates"),

        F.round(
            F.avg("delivery_days"), 2
        ).alias("average_delivery_days"),

        F.sum(
            F.when(
                F.col("is_late_delivery") == True, 1
            ).otherwise(0)
        ).alias("late_orders"),

        F.sum(
            F.when(
                (F.col("order_status") == "canceled") &
                F.col("delivery_days").isNotNull(), 1
            ).otherwise(0)
        ).alias("cancelled_with_delivery_dates")
    )
)

(
    delivery_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.delivery_performance")
)

display(delivery_performance)


# Save analytics tables
analytics_tables = {
    "daily_sales": daily_sales,
    "seller_performance": seller_performance,
    "delivery_performance": delivery_performance
}

for name, df in analytics_tables.items():
    (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.gold.{name}")
    )

print("Business analytics tables created!")

order_status,total_orders,orders_with_delivery_dates,orders_without_delivery_dates,average_delivery_days,late_orders,cancelled_with_delivery_dates
delivered,96478,96470,8,12.5,6534,0
processing,301,0,301,null,0,0
shipped,1107,0,1107,null,0,0
canceled,625,6,619,20.33,1,6
invoiced,314,0,314,null,0,0
unavailable,609,0,609,null,0,0
created,5,0,5,null,0,0
approved,2,0,2,null,0,0


Business analytics tables created!


In [0]:
gold_tables = [
    "dim_customers",
    "dim_products",
    "dim_sellers",
    "dim_date",
    "fact_orders",
    "fact_order_items",
    "daily_sales",
    "seller_performance",
    "delivery_performance"
]

for table in gold_tables:
    count = spark.table(f"workspace.gold.{table}").count()
    print(f"{table}: {count} records")

dim_customers: 99441 records
dim_products: 32951 records
dim_sellers: 3095 records
dim_date: 634 records
fact_orders: 99441 records
fact_order_items: 112650 records
daily_sales: 634 records
seller_performance: 3095 records
delivery_performance: 8 records


In [0]:
display(
    spark.table("workspace.gold.delivery_performance")
)

order_status,total_orders,orders_with_delivery_dates,orders_without_delivery_dates,average_delivery_days,late_orders,cancelled_with_delivery_dates
invoiced,314,0,314,null,0,0
processing,301,0,301,null,0,0
shipped,1107,0,1107,null,0,0
unavailable,609,0,609,null,0,0
created,5,0,5,null,0,0
approved,2,0,2,null,0,0
delivered,96478,96470,8,12.5,6534,0
canceled,625,6,619,20.33,1,6
